# OSRS NPC Classification
# Model training

Train a CNN to classify NPC race/class from chathead + body images

- **Input:** chathead image + body image (both passed through separate CNN backbones, then features concatenated)
- **Target:** Class (top 10 classes in the data, with all others combined to form "Other")
- **Split:** 70/15/15 train/val/test
- **Augmentation:** applied to minority classes to address class imbalance; used heavily in some cases

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import zipfile
with zipfile.ZipFile('/content/drive/MyDrive/Computer Vision/cv-final-clean.zip', 'r') as z:
    z.extractall('/content/cv-final')

Mounted at /content/drive


In [2]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

In [3]:
df = pd.read_csv('/content/cv-final/data/npc.csv')

# cleaning (should mirror cleaning we wrote in the EDA)
# TODO: EDA notebook should output a cleaned csv that this model uses and then we skip cleaning here
df["Gender"] = df["Gender"].apply(lambda x: x if x in ("Male", "Female") else "Other")
df["Members"] = df["Members"].replace("? (edit)", "Yes")
df = df.rename(columns={"Race": "Class"})
df["Class"] = df["Class"].replace({
    "Citizen of Arceuus": "Human",
    "Dwarf ( Imcando-descendant )": "Dwarf"
})
df["has_chathead"] = df["id"].apply(lambda i: os.path.exists(f"/content/cv-final/data/chatheads/{i}.png"))
df["has_body"] = df["id"].apply(lambda i: os.path.exists(f"/content/cv-final/data/bodies/{i}.png"))

# one img corrupted
# TODO: what is this? how did it get corrupted? fixable?
corrupt = {4231}
df = df[~df["id"].isin(corrupt)].reset_index(drop=True)

# filter to NPCs with both images, since the CNN will use both images in its inference
# TODO: should we let the model still make a prediction on NPCs even if the chathead is missing? --> yes!
df = df[df["has_chathead"] & df["has_body"]].reset_index(drop=True)
print(f"Working dataset: {len(df)} NPCs")

Working dataset: 3467 NPCs


In [4]:
df["Class"].value_counts().head(10)

,count
Class,
Human,2097
Dwarf,248
Elf,149
Gnome,139
Vampyre,60
Ghost,49
Monkey,42
Dorgeshuun,40
Troll,31


In [5]:
# collapse to top 10 classes + Other
TOP_CLASSES = df["Class"].value_counts().head(10).index.tolist()
df["label"] = df["Class"].apply(lambda x: x if x in TOP_CLASSES else "Other")

CLASS_NAMES = TOP_CLASSES + ["Other"]
class_to_idx = {c: i for i, c in enumerate(CLASS_NAMES)}
df["label_idx"] = df["label"].map(class_to_idx)

print(df["label"].value_counts())

label
Human         2097
Other          582
Dwarf          248
Elf            149
Gnome          139
Vampyre         60
Ghost           49
Monkey          42
Dorgeshuun      40
Troll           31
Cat             30
Name: count, dtype: int64


# Split data into training/val/test

In [6]:
# 70/15/15 split, stratified by label
train_df, temp_df = train_test_split(df, test_size=0.30, stratify=df["label_idx"], random_state=42)
val_df, test_df = train_test_split(temp_df, test_size=0.50, stratify=temp_df["label_idx"], random_state=42)

print(f"Train: {len(train_df)} | Val: {len(val_df)} | Test: {len(test_df)}")

Train: 2426 | Val: 520 | Test: 521


## Augmentation

We have class imbalance. And some of the classes get very thin. Cat/Troll both have ~30 NPCs in the sample only. Meanwhile we have over 2,000 humans.

To get around this, we'll use data augmentation during training (but not val/test).

**All training images:**
- Random horizontal flip (p=0.3) to add variety
- Random rotation (±15°)
- Color jitter (brightness/contrast ±0.3) to help generalize across lighting variations in pixels

**Minority classes (everything except Human and Other):**
- Random affine transform (rotation, translation, scale) to simulates slight pose variation
- Random perspective distortion (p=0.5) to adda viewpoint robustness

The heavier augmentation on minority classes basically oversamples them, giving the model more varied examples of rare classes like Vampyre, Cat, and Troll without duplicating identical images.

In [7]:
# augmentation for training (helps minority classes)
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.3),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.3, contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

# no augmentation for val/test
eval_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [15]:
class NPCDataset(Dataset):
    def __init__(self, df, transform):
        self.df = df.reset_index(drop=True)
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        chathead = Image.open(f"/content/cv-final/data/chatheads/{row['id']}.png").convert("RGB")
        body = Image.open(f"/content/cv-final/data/bodies/{row['id']}.png").convert("RGB")
        # heavier augmentation for minority classes (anything below Human/Other threshold)
        if self.transform == train_transform and row["label"] not in ["Human", "Other"]:
            extra = transforms.Compose([
                transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.85, 1.15)),
                transforms.RandomPerspective(distortion_scale=0.2, p=0.5),
            ])
            chathead = extra(chathead)
            body = extra(body)
        return self.transform(chathead), self.transform(body), row["label_idx"]

class DualCNN(nn.Module):
    def __init__(self, num_classes):
        super().__init__()
        # two separate resnet18 backbones, one for chathead one for body
        self.chathead_cnn = models.resnet18(weights="IMAGENET1K_V1")
        self.body_cnn = models.resnet18(weights="IMAGENET1K_V1")
        # remove final classification layer from both
        self.chathead_cnn.fc = nn.Identity()
        self.body_cnn.fc = nn.Identity()
        # concatenate both 512-dim feature vectors -> classify
        self.classifier = nn.Sequential(
            nn.Linear(512 * 2, 256),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(256, num_classes)
        )

    def forward(self, chathead, body):
        f1 = self.chathead_cnn(chathead)
        f2 = self.body_cnn(body)
        return self.classifier(torch.cat([f1, f2], dim=1))

def run_epoch(loader, train=True):
    model.train() if train else model.eval()
    total_loss, correct, total = 0, 0, 0
    with torch.set_grad_enabled(train):
        for chathead, body, labels in loader:
            chathead, body, labels = chathead.to(device), body.to(device), labels.to(device)
            preds = model(chathead, body)
            loss = criterion(preds, labels)
            if train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()
            total_loss += loss.item()
            correct += (preds.argmax(1) == labels).sum().item()
            total += len(labels)
    return total_loss / len(loader), correct / total

## Model Architecture

### Backbone
Two separate ResNet18 backbones pretrained on ImageNet — one for chathead images, one for body images. The final classification layer of each is replaced with `nn.Identity()` to extract 512-dim feature vectors.

### Fusion
The two 512-dim feature vectors are concatenated into a single 1024-dim vector, then passed through a classification head:
- Linear(1024 → 256)
- ReLU
- Dropout(0.5) — increased from 0.3 to reduce overfitting on tail classes
- Linear(256 → n_classes)

### Training
- **Loss:** CrossEntropyLoss with class weights (inverse frequency) to further penalize errors on minority classes
- **Optimizer:** Adam, lr=5e-5 (reduced from 1e-4 for finer convergence with 10 classes)
- **Scheduler:** ReduceLROnPlateau — halves LR if val loss doesn't improve for 2 epochs
- **Early stopping:** patience=3 on val loss, saves best checkpoint
- **Input size:** 128×128 for both chathead and body

In [16]:
train_loader = DataLoader(NPCDataset(train_df, train_transform), batch_size=32, shuffle=True)
val_loader = DataLoader(NPCDataset(val_df, eval_transform), batch_size=32)
test_loader = DataLoader(NPCDataset(test_df, eval_transform), batch_size=32)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using: {device}")
model = DualCNN(num_classes=len(CLASS_NAMES)).to(device)

# class weights to further help with imbalance
counts = df["label_idx"].value_counts().sort_index().values
weights = torch.tensor(1.0 / counts, dtype=torch.float).to(device)

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.Adam(model.parameters(), lr=5e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=2, factor=0.5)

Using: cuda


In [17]:
best_val_loss = float('inf')
patience = 3
epochs_no_improve = 0

EPOCHS = 20
for epoch in range(EPOCHS):
    train_loss, train_acc = run_epoch(train_loader, train=True)
    val_loss, val_acc = run_epoch(val_loader, train=False)
    print(f"Epoch {epoch+1}/{EPOCHS} | train loss: {train_loss:.3f} acc: {train_acc:.3f} | val loss: {val_loss:.3f} acc: {val_acc:.3f}")
    scheduler.step(val_loss)
    if val_loss < best_val_loss:
        best_val_loss = val_loss
        torch.save(model.state_dict(), '/content/best_model.pth')
        epochs_no_improve = 0
        print("  ✓ saved best model")
    else:
        epochs_no_improve += 1
        if epochs_no_improve >= patience:
            print("Early stopping.")
            break

/usr/local/lib/python3.12/dist-packages/PIL/Image.py:1047: UserWarning: Palette images with Transparency expressed in bytes should be converted to RGBA images
  warnings.warn(


Epoch 1/20 | train loss: 2.036 acc: 0.264 | val loss: 1.410 acc: 0.698
  ✓ saved best model
Epoch 2/20 | train loss: 1.022 acc: 0.661 | val loss: 0.772 acc: 0.790
  ✓ saved best model
Epoch 3/20 | train loss: 0.503 acc: 0.788 | val loss: 0.614 acc: 0.815
  ✓ saved best model
Epoch 4/20 | train loss: 0.338 acc: 0.848 | val loss: 0.606 acc: 0.835
  ✓ saved best model
Epoch 5/20 | train loss: 0.237 acc: 0.887 | val loss: 0.461 acc: 0.883
  ✓ saved best model
Epoch 6/20 | train loss: 0.173 acc: 0.908 | val loss: 0.498 acc: 0.888
Epoch 7/20 | train loss: 0.151 acc: 0.917 | val loss: 0.470 acc: 0.904
Epoch 8/20 | train loss: 0.132 acc: 0.934 | val loss: 0.557 acc: 0.898
Early stopping.


In [18]:
model.load_state_dict(torch.load('/content/best_model.pth'))

# final test evaluation (we do much more eval in the next notebook of the pipeline)
test_loss, test_acc = run_epoch(test_loader, train=False)
print(f"Test accuracy: {test_acc:.3f}")

Test accuracy: 0.862


In [20]:
# save model to drive
torch.save(model.state_dict(), '/content/drive/MyDrive/Computer Vision/npc_classifier.pth')